# ⚡ Smart Power — Final Project (Day 1 Setup)

**Question:** When is electricity in the Netherlands cheapest **and** cleanest, and how much is driven by renewable generation? → a practical recommendation on the best hours to use power.

**This notebook = Day 1:** get the data sources working. Everything in English (this repo goes on GitHub / portfolio).

## ✅ Today's checklist

- [ ] Run **Data Source 1** (Open-Meteo) — it works if it prints values
- [ ] Turn it into a clean **pandas** table
- [ ] Run **Data Source 2** (EnergyZero electricity prices) — test it
- [ ] Register at transparency.entsoe.eu **+ email for API token** (arrives in up to 3 working days)
- [ ] Send project **scope to Adi + Miguel** on Slack
- [ ] Read the **5-minute pitch** out loud once (~5 min)

> Goal for today: scope is locked + at least one data source works + ENTSO-E token requested. That's it.

## Data Source 1 — Open-Meteo (renewables proxy, no API key)

Wind speed and solar radiation are a proxy for how much renewable energy is available.

In [ ]:
import requests

# Amsterdam: wind speed + solar radiation (renewable-energy proxy)
url = "https://api.open-meteo.com/v1/forecast"
params = {
    "latitude": 52.37,
    "longitude": 4.90,
    "hourly": "wind_speed_10m,shortwave_radiation",
}

response = requests.get(url, params=params)
data = response.json()

# Quick check: first 5 hourly timestamps and wind values
print(data["hourly"]["time"][:5])
print(data["hourly"]["wind_speed_10m"][:5])

In [ ]:
import pandas as pd

# Turn the API response into a clean table
weather_df = pd.DataFrame({
    "timestamp": data["hourly"]["time"],
    "wind_speed": data["hourly"]["wind_speed_10m"],
    "solar_radiation": data["hourly"]["shortwave_radiation"],
})
weather_df["timestamp"] = pd.to_datetime(weather_df["timestamp"])

print(weather_df.shape)
weather_df.head()

## Data Source 2 — Dutch electricity prices (EnergyZero, no API key)

EnergyZero exposes the hourly Dutch day-ahead electricity prices. No token needed.

> Run the cell and look at the printed keys first. If the field names are different from what we expect, tell me the printed structure and we'll adjust the parser.

In [ ]:
from datetime import datetime, timedelta, timezone

# Use yesterday (a full day of prices is available)
day = (datetime.now(timezone.utc) - timedelta(days=1)).strftime("%Y-%m-%d")

price_url = "https://api.energyzero.nl/v1/energyprices"
price_params = {
    "fromDate": f"{day}T00:00:00.000Z",
    "tillDate": f"{day}T23:59:59.999Z",
    "interval": 4,       # hourly
    "usageType": 1,      # electricity
    "inclBtw": "true",   # include VAT
}

price_resp = requests.get(price_url, params=price_params)
price_data = price_resp.json()

# Defensive check: see the structure before parsing
print("Top-level keys:", list(price_data.keys()))
print("First price entry:", price_data.get("Prices", [{}])[0])

In [ ]:
# Build a clean price table (adjust keys if the printout above differs)
prices_df = pd.DataFrame(price_data["Prices"])
prices_df = prices_df.rename(columns={"readingDate": "timestamp", "price": "electricity_price"})
prices_df["timestamp"] = pd.to_datetime(prices_df["timestamp"])

print(prices_df.shape)
prices_df.head()

## Data Source 3 (upgrade) — ENTSO-E (needs token, requested today)

ENTSO-E gives richer price + actual generation-by-source data. It needs a security token,
which can take up to 3 working days. **We requested it today** — once it arrives we plug it in here.

Until then, the MVP runs fine on Open-Meteo + EnergyZero above.

## Next steps (this week)

1. **Join** prices + renewables on `timestamp` (one combined hourly table)
2. **EDA:** average electricity price by hour of day / day of week; correlation renewables ↔ price
3. **Add carbon** intensity (Electricity Maps API) → find cheapest + cleanest hours
4. **Dashboard** (Streamlit or Tableau) → "best hours to use electricity"
5. **Automate** the pipeline to run daily (GitHub Actions)

Keep the MVP small: one API + one clean table + one dashboard insight = a complete project.

In [ ]:
!pip install entsoe-py

In [ ]:
import os
from dotenv import load_dotenv
from entsoe import EntsoePandasClient
import pandas as pd

load_dotenv()  # reads ENTSOE_API_TOKEN from your .env file (never hard-code the token)
client = EntsoePandasClient(api_key=os.getenv("ENTSOE_API_TOKEN"))

start = pd.Timestamp("20260801", tz="Europe/Amsterdam")
end   = pd.Timestamp("20260803", tz="Europe/Amsterdam")

# Day-ahead electricity prices for the Netherlands
prices = client.query_day_ahead_prices("NL", start=start, end=end)
print(prices.head())

# Actual generation per production type (renewable share!)
generation = client.query_generation("NL", start=start, end=end)
print(generation.head())

## Transformation — resample generation to hourly + renewable share

Now we work from the raw CSVs that `src/ingest.py` produced (90 days, all in UTC).
Generation is 15-min, so we resample it to hourly and compute the **renewable share**
for every hour — the key variable of the whole project.

In [ ]:
import pandas as pd

# Load the raw data produced by src/ingest.py
weather = pd.read_csv("../data/raw_weather.csv")
prices  = pd.read_csv("../data/raw_prices.csv")
gen     = pd.read_csv("../data/raw_generation.csv")

# Make sure every timestamp is a proper UTC datetime
for name, df in [("weather", weather), ("prices", prices), ("generation", gen)]:
    df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)
    print(name, df.shape)

In [ ]:
# Resample generation from 15-min to hourly (average power per hour)
gen = gen.set_index("timestamp")
gen_hourly = gen.resample("h").mean()

print(gen_hourly.shape)   # ~2160 rows = 90 days x 24 hours
gen_hourly.head()

In [ ]:
# Compute the renewable share for every hour
production_cols = gen_hourly.columns.tolist()          # all generation types

# Pick renewable columns by name (Wind Offshore, Wind Onshore, Solar, Hydro..., Biomass)
renewable_keywords = ["Solar", "Wind", "Hydro", "Biomass"]
renewable_cols = [c for c in production_cols
                  if any(k in c for k in renewable_keywords)]
print("Renewable columns:", renewable_cols)

gen_hourly["total_generation"]     = gen_hourly[production_cols].sum(axis=1)
gen_hourly["renewable_generation"] = gen_hourly[renewable_cols].sum(axis=1)
gen_hourly["renewable_share"]      = (
    gen_hourly["renewable_generation"] / gen_hourly["total_generation"]
)

gen_hourly[["total_generation", "renewable_generation", "renewable_share"]].head()

In [ ]:
# Sanity check: renewable_share should sit between 0 and 1
print(gen_hourly["renewable_share"].describe())